# Construction et Optimisation du Moteur de Prédiction

## Objectif

Cette étape a pour but d'entraîner, d'évaluer et d'optimiser différents algorithmes de Machine Learning afin de sélectionner le modèle le plus performant pour prédire le rendement agricole (`Yield_tons_per_hectare`).

La démarche suit les standards MLOps, incluant un suivi rigoureux des expérimentations, et se déroulera selon les étapes suivantes :

- **Préparation des données** : Séparation du jeu de données en ensembles d'entraînement et de test (*Train/Test split*).
- **Pipeline de pré-traitement** : Standardisation des variables numériques (*StandardScaler*) et encodage des variables catégorielles (*One-Hot Encoding*).
- **Expérimentations et Tracking** : Entraînement de plusieurs modèles de régression (ex: Random Forest, XGBoost) avec journalisation systématique via **MLflow**.
- **Évaluation des performances** : Analyse des modèles à l'aide des métriques de régression (RMSE et R²).
- **Optimisation** : Recherche des meilleurs hyperparamètres (Fine-tuning) pour le modèle retenu.
- **Sérialisation** : Sauvegarde (*persistance*) du modèle final prêt à être déployé en production via l'API.

## Préparation des données

In [11]:
import pandas as pd

#Importation du dataframe
df = pd.read_csv("../data/processed/dataset_ml_ready.csv")

display(df.head(5))


,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare,pesticides_tonnes_mean
0,West,Sandy,Cotton,897.077239,27.676966,False,True,Cloudy,122,6.555816,0.000000
1,South,Clay,Rice,992.673282,18.026142,True,True,Rainy,140,8.527341,36942.215995
2,North,Loam,Barley,147.998025,29.794042,False,False,Sunny,106,1.127443,0.000000
3,North,Sandy,Soybean,986.866331,16.644190,False,True,Rainy,146,6.517573,40752.554896
4,South,Silt,Wheat,730.379174,31.620687,True,True,Cloudy,110,7.248251,35453.212930


In [12]:
from sklearn.model_selection import train_test_split

#Séparation des données
y = df["Yield_tons_per_hectare"]
X = df.drop(columns=["Yield_tons_per_hectare"])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train")
display(X_train.head())
print("X_test")
display(X_test.head())
print("y_train")
display(y_train.head())
print("y_test")
display(y_test.head())



X_train


,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,pesticides_tonnes_mean
816226,West,Clay,Soybean,835.288831,39.936159,True,False,Cloudy,77,40752.554896
180530,North,Sandy,Barley,867.649855,35.125782,True,False,Sunny,131,0.000000
765928,North,Peaty,Cotton,269.615635,19.801171,True,True,Cloudy,67,0.000000
207782,North,Chalky,Wheat,923.607497,21.427055,False,False,Rainy,128,35453.212930
411529,North,Loam,Soybean,354.781067,18.384813,True,True,Cloudy,96,40752.554896


X_test


,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,pesticides_tonnes_mean
352142,West,Peaty,Rice,423.756311,39.298567,False,True,Cloudy,71,36942.215995
30783,South,Loam,Barley,156.270440,20.391569,False,False,Sunny,85,0.000000
479365,North,Silt,Soybean,914.253783,34.803208,True,True,Sunny,120,40752.554896
55589,North,Silt,Soybean,856.558093,35.352312,False,True,Cloudy,102,40752.554896
493689,East,Silt,Maize,796.598674,15.957662,True,True,Cloudy,83,32765.983322


y_train


816226    6.411219
180530    7.096720
765928    4.404192
207782    5.364438
411529    4.768660
Name: Yield_tons_per_hectare, dtype: float64

y_test


352142    4.378713
30783     1.589250
479365    8.032276
55589     6.163982
493689    6.424947
Name: Yield_tons_per_hectare, dtype: float64

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

#Encodage des variables catégorielles 
numeric_features =[
    "Rainfall_mm",
    "Temperature_Celsius",
    "Days_to_Harvest",
    "pesticides_tonnes_mean",
]

categorical_features = [
    "Region",
    "Soil_Type",
    "Crop",
    "Weather_Condition",
]

boolean_features = ["Fertilizer_Used", "Irrigation_Used"]

categorical_transformer = OneHotEncoder(
    drop="first",
    handle_unknown="ignore"
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        ),
        (
            "bool",
            "passthrough",
            boolean_features
        )
    ]
)


# preprocessor.fit(X_train)

# X_train_transformed = preprocessor.transform(X_train)
# X_test_transformed = preprocessor.transform(X_test)


## Exprériences MLFLOW

In [14]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

print(mlflow.get_tracking_uri())

mlflow.set_experiment("Agritech_Yield_Prediction")

http://127.0.0.1:5000


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1789572878110, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789572878110, lifecycle_stage='active', name='Agritech_Yield_Prediction', tags={}, trace_location=None, workspace='default'>

In [15]:
#Construction du pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

regressor = RandomForestRegressor(random_state=42)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", regressor)
])


In [16]:
from sklearn.metrics import root_mean_squared_error, r2_score

with mlflow.start_run(run_name="RandomForestRegressor"):
    #Entraînement
    pipeline.fit(X_train, y_train)

    #Prédictions
    y_pred = pipeline.predict(X_test)

    # Évaluation
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    #Affichage dans le notebook
    print("Modèle entraîné avec succès !")
    print(f"RMSE : {rmse:.3f}")
    print(f"R² : {r2:.3f}")

    #Logging dans MLflow
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.log_param("model", "RandomForestRegressor")

Modèle entraîné avec succès !
RMSE : 0.515
R² : 0.908
🏃 View run RandomForestRegressor at: http://127.0.0.1:5000/#/experiments/1/runs/06931df94a7b433d9e1c3cbc9ab3bed9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [17]:
display(df["Yield_tons_per_hectare"].max())
display(df["Yield_tons_per_hectare"].min())

np.float64(9.963372228814649)

np.float64(0.0004108724039286)

In [18]:
(df["Yield_tons_per_hectare"] < 0).sum()
#display(df["Yield_tons_per_hectare"].describe())

np.int64(0)

### Interprétation des résultats avec **RandomForestRegressor**

Le modèle obtient un **RMSE de 0,515 tonne/ha** et un **R² de 0,908** sur le jeu de test.

Ces résultats indiquent que le modèle parvient à prédire les rendements avec un écart typique d'environ **0,52 tonne par hectare**, alors que les rendements observés après nettoyage s'étendent d'environ **0 à 9,96 tonnes par hectare**.

Le **R² de 0,908** signifie que le modèle explique environ **90,8 % de la variabilité des rendements** observée sur le jeu de test. Les variations restantes, soit environ 9,2 %, ne sont pas expliquées par le modèle et peuvent notamment être liées à des facteurs absents des données ou à une part de variabilité difficile à prédire.

Les performances obtenues constituent ainsi une **baseline de référence** pour la suite du projet. Les futurs modèles pourront être comparés à ces résultats afin d'évaluer si les modifications apportées au modèle ou aux données permettent de réduire le RMSE et/ou d'augmenter le R².

---

Nous allons faire le deuxième modèle avec **HistGradientBoostingRegressor**, en gardant exactement la même préparation des données et le même jeu de test que pour ke Random Forest.

In [22]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
import time

hgb_model = HistGradientBoostingRegressor(random_state=42)

hgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", hgb_model),
])

with mlflow.start_run(run_name="HistGradientBoosting_Baseline_wtime"):
    #Entrainement
    start_train = time.perf_counter()
    hgb_pipeline.fit(X_train, y_train)
    train_time = time.perf_counter() - start_train

    #Prédiction
    start_pred = time.perf_counter()
    y_pred_hgb = hgb_pipeline.predict(X_test)
    pred_time  = time.perf_counter() - start_pred

    rmse_hgb = root_mean_squared_error(y_test, y_pred_hgb)
    r2_hgb = r2_score(y_test, y_pred_hgb)

    print("HistGradientBoostingRegressor")
    print(f"RMSE : {rmse_hgb:.3f}")
    print(f"R² : {r2_hgb:.3f}")
    print(f"Temps entraînement : {train_time:.2f} s")
    print(f"Temps prédiction   : {pred_time:.2f} s")
    print(f"Temps total        : {train_time + pred_time:.2f} s")

    #Logging dans MLflow
    mlflow.log_metric("rmse", rmse_hgb)
    mlflow.log_metric("r2", r2_hgb)
    mlflow.log_param("model", "HistGradientBoostingRegressor")
    mlflow.log_metric("train_time_seconds", train_time)
    mlflow.log_metric("prediction_time_seconds", pred_time)
    mlflow.log_metric("total_time_seconds", train_time + pred_time)


HistGradientBoostingRegressor
RMSE : 0.500
R² : 0.913
Temps entraînement : 6.46 s
Temps prédiction   : 0.40 s
Temps total        : 6.86 s
🏃 View run HistGradientBoosting_Baseline_wtime at: http://127.0.0.1:5000/#/experiments/1/runs/05dfc5dcb7db412a8df3bf0132a13a36
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Nous allons faire le deuxième modèle avec **XGBoost**, en gardant exactement la même préparation des données et le même jeu de test que pour ke Random Forest.

In [25]:
from xgboost import XGBRegressor
import time

xgb_model = XGBRegressor(
    n_estimators=300, #nombre d'arbres
    learning_rate=0.05, #vitesse d'apprentissage(correction à chaque nouvel arbre)
    max_depth=6, #complexité des arbres
    random_state=42, #reproductibilité
    n_jobs=-1 #utilisation du CPU
)

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", xgb_model)
])

with mlflow.start_run(run_name="XGBRegressor_wtime"):
    #Entrainement
    start_train = time.perf_counter()
    xgb_pipeline.fit(X_train, y_train)
    train_time = time.perf_counter() - start_train

    #Prédiction
    start_pred = time.perf_counter()
    y_pred_xgb = xgb_pipeline.predict(X_test)
    pred_time  = time.perf_counter() - start_pred


    #Évaluation
    rmse_xgb = root_mean_squared_error(y_test, y_pred_xgb)
    r_2_xgb = r2_score(y_test, y_pred_xgb)

    print("XGBRegressor")
    print(f"RMSE : {rmse_xgb:.3f}")
    print(f"R² : {r_2_xgb:.3f}")
    print(f"Temps entraînement : {train_time:.2f} s")
    print(f"Temps prédiction   : {pred_time:.2f} s")
    print(f"Temps total        : {train_time + pred_time:.2f} s")

    #Logging dans MLflow
    mlflow.log_metric("rmse", rmse_xgb)
    mlflow.log_metric("r2", r_2_xgb)
    
    mlflow.log_param("model", "XGBRegressor")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)

    mlflow.log_metric("train_time_seconds", train_time)
    mlflow.log_metric("prediction_time_seconds", pred_time)
    mlflow.log_metric("total_time_seconds", train_time + pred_time)



XGBRegressor
RMSE : 0.500
R² : 0.913
Temps entraînement : 5.77 s
Temps prédiction   : 0.28 s
Temps total        : 6.05 s
🏃 View run XGBRegressor_wtime at: http://127.0.0.1:5000/#/experiments/1/runs/dc91f558d13d4b8a80e73f2a17b17c52
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Comparaison des modèles

Trois modèles ont été entraînés avec le même jeu de données et évalués sur le même jeu de test.

| Modèle | RMSE (↓) | R² (↑) | Temps d'entraînement (↓) | Temps de prédiction (↓) | Temps total (↓) |
|---|---:|---:|---:|---:|---:|
| Random Forest | 0,515 | 0,908 | — | — | — |
| HistGradientBoosting | 0,500 | 0,913 | 6,46 s | 0,40 s | 6,86 s |
| XGBoost | 0,500 | 0,913 | 5,77 s | 0,28 s | 6,05 s |

### Analyse

Les deux modèles de Gradient Boosting obtiennent de meilleures performances que le Random Forest, avec un **RMSE de 0,500 tonne/ha** et un **R² de 0,913**.

À trois décimales, HistGradientBoosting et XGBoost présentent donc des performances équivalentes.

En revanche, **XGBoost est légèrement plus rapide** que HistGradientBoosting :

- Temps d'entraînement : **5,77 s** contre 6,46 s.
- Temps de prédiction : **0,28 s** contre 0,40 s.
- Temps total : **6,05 s** contre 6,86 s.

### Conclusion

Le **Random Forest est conservé comme baseline de référence**.

Entre HistGradientBoosting et XGBoost, les performances obtenues sont équivalentes sur les métriques évaluées. Le temps d'exécution constitue donc un critère supplémentaire pour les départager.

Avec un temps total de **6,05 secondes contre 6,86 secondes**, **XGBoost est retenu pour la phase d'optimisation**.

L'objectif sera maintenant d'optimiser ses hyperparamètres afin de chercher à améliorer ses performances tout en surveillant le temps d'exécution.

## Recherche des paramètres optimisés

In [34]:
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
import mlflow

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", xgb_model)
])

param_distributions = {
    # Nombre d'arbres : teste de 200 à 700 arbres
    "regressor__n_estimators": [200, 300, 400, 500, 700],

    # Taux d'apprentissage : contribution de chaque arbre
    "regressor__learning_rate": [0.01, 0.03, 0.05, 0.1],

    # Seuil minimum pour permettre une nouvelle séparation dans un arbre
    "regressor__min_child_weight": [1, 3, 5, 7],

    # Proportion des données utilisée pour construire chaque arbre
    "regressor__subsample": [0.7, 0.8, 0.9, 1.0],
}

random_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=15,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=42,
    n_jobs=1,
    verbose=1
)

with mlflow.start_run(run_name="RandomizedSearchCV"):
    random_search.fit(X_train, y_train)

    best_score = f"{-random_search.best_score_:.6f}"
    best_params = random_search.best_params_
    #Logging
    mlflow.log_metric("Meilleur RMSE CV", best_score)
    mlflow.log_param("model", "XGBRegressor")
    mlflow.log_param("regressor__subsample", {best_params["regressor__subsample"]})
    mlflow.log_param("regressor__n_estimators", {best_params["regressor__n_estimators"]})
    mlflow.log_param("regressor__min_child_weight", {best_params["regressor__min_child_weight"]})
    mlflow.log_param("regressor__learning_rate", {best_params["regressor__learning_rate"]})

    print("Meilleurs paramètres :")
    print(best_params)
    print(f"\nMeilleur RMSE CV : {best_score}")


Fitting 3 folds for each of 15 candidates, totalling 45 fits
Meilleurs paramètres :
{'regressor__subsample': 0.7, 'regressor__n_estimators': 700, 'regressor__min_child_weight': 7, 'regressor__learning_rate': 0.01}

Meilleur RMSE CV : 0.500818
🏃 View run RandomizedSearchCV at: http://127.0.0.1:5000/#/experiments/1/runs/38ade7db5a434636827876bcba95f9c7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [31]:
#Evaluation finale
best_xgb_pipeline = random_search.best_estimator_
from sklearn.metrics import root_mean_squared_error, r2_score

y_pred = best_xgb_pipeline.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("XGBoost optimisé")
print(f"RMSE : {rmse:.6f}")
print(f"R²   : {r2:.6f}")

XGBoost optimisé
RMSE : 0.499595
R²   : 0.913121


In [32]:
from sklearn.metrics import root_mean_squared_error, r2_score

#Modèle optimisé
best_xgb_pipeline = random_search.best_estimator_

#Prédictions sur le jeu d'entraînement
y_train_pred = best_xgb_pipeline.predict(X_train)

#Prédictions sur le jeu de test
y_test_pred = best_xgb_pipeline.predict(X_test)

#Métriques train
rmse_train = root_mean_squared_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)

#Métriques test
rmse_test = root_mean_squared_error(y_test, y_test_pred)
r2_test = r2_score(y_test, y_test_pred)

print("XGBoost optimisé - comparaison Train / Test")
print("-" * 50)

print(f"RMSE Train : {rmse_train:.6f}")
print(f"R² Train   : {r2_train:.6f}")

print()

print(f"RMSE Test  : {rmse_test:.6f}")
print(f"R² Test    : {r2_test:.6f}")

XGBoost optimisé - comparaison Train / Test
--------------------------------------------------
RMSE Train : 0.499174
R² Train   : 0.913293

RMSE Test  : 0.499595
R² Test    : 0.913121


### Vérification du surapprentissage

Afin de vérifier la capacité de généralisation du modèle optimisé, nous comparons ses performances sur les jeux d'entraînement et de test.

| Métrique | Train | Test |
|---|---:|---:|
| RMSE | 0,499174 | 0,499595 |
| R² | 0,913293 | 0,913121 |

Les performances obtenues sur les deux jeux de données sont très proches. L'écart de RMSE est de seulement **0,000421 tonne/ha**, tandis que l'écart de R² est de **0,000172**.

Cette faible différence ne met pas en évidence de surapprentissage important. Le modèle semble ainsi conserver des performances similaires sur des données qu'il n'a pas utilisées lors de son entraînement.

Le modèle optimisé peut donc être conservé pour la suite de l'analyse.

In [36]:
import joblib
import os

# Création du dossier s'il n'existe pas
os.makedirs("../models", exist_ok=True)

# Sauvegarde du modèle final (le pipeline complet qui inclut le preprocessor !)
joblib.dump(best_xgb_pipeline, "../models/best_xgb_pipeline.pkl")

print("✅ Modèle sauvegardé avec succès dans ../models/best_xgb_pipeline.pkl")

✅ Modèle sauvegardé avec succès dans ../models/best_xgb_pipeline.pkl


## Conclusion — Évaluation et optimisation du modèle

Plusieurs modèles de régression ont été évalués afin de prédire le rendement agricole :

- **Random Forest** : RMSE = 0,515 tonne/ha et R² = 0,908.
- **HistGradientBoosting** : RMSE = 0,500 tonne/ha et R² = 0,913.
- **XGBoost** : RMSE = 0,500 tonne/ha et R² = 0,913.

Les modèles de Gradient Boosting obtiennent de meilleures performances que le Random Forest sur le jeu de test. HistGradientBoosting et XGBoost présentent des performances très proches, mais XGBoost possède un temps d'exécution légèrement inférieur dans les expérimentations réalisées.

XGBoost a donc été retenu pour une phase d'optimisation avec `RandomizedSearchCV`. Parmi les combinaisons d'hyperparamètres testées, la meilleure configuration obtenue est :

- `n_estimators` : 700
- `learning_rate` : 0,01
- `min_child_weight` : 7
- `subsample` : 0,7

Après optimisation, le modèle obtient :

- **RMSE : 0,499595 tonne/ha**
- **R² : 0,913121**

L'amélioration par rapport au modèle XGBoost initial reste faible, ce qui indique que le modèle baseline était déjà performant sur ce jeu de données. L'optimisation permet néanmoins d'obtenir une légère amélioration des métriques.

Une comparaison des performances sur les jeux d'entraînement et de test a également été réalisée :

| Métrique | Train | Test |
|---|---:|---:|
| RMSE | 0,499174 | 0,499595 |
| R² | 0,913293 | 0,913121 |

Les performances étant très proches entre les deux jeux de données, cette comparaison ne met pas en évidence de surapprentissage important.

Enfin, le **pipeline complet** contenant à la fois le préprocesseur et le modèle XGBoost optimisé a été sauvegardé au format `joblib`. Cette sauvegarde permet de conserver les étapes de prétraitement avec le modèle et de pouvoir réutiliser directement le pipeline lors de futures prédictions.

Le modèle final est enregistré dans :

`../models/best_xgb_pipeline.pkl`

Cette étape clôt la phase de modélisation et prépare la suite du projet, notamment la **réutilisation du modèle pour effectuer des prédictions sur de nouvelles données via API**.